# A11. SKU 真实利润计算器 Notebook

> **配套模块**: [A11 财务分析](../paths/a-operators/a11-financial-analysis.md)
>
> **功能**: 输入成本数据 → 自动计算真实利润 + 成本分解 + 敏感度分析
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/a11-profit-calculator.ipynb)

---

## 1. 安装依赖

In [ ]:
!pip install -q pandas numpy plotly

## 2. 输入产品数据

替换下面的数据为你的真实数据。

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# === 替换为你的真实数据 ===
products = [
    {
        'sku': 'SKU-001',
        'name': 'Wireless Earbuds Pro',
        'selling_price': 39.99,
        'monthly_units': 450,
        'cogs_per_unit': 8.50,        # 采购成本 (FOB)
        'shipping_per_unit': 1.20,     # 国际运费/件
        'tariff_rate': 0.05,           # 关税率 5%
        'referral_fee_rate': 0.15,     # Amazon 佣金 15%
        'fba_fee_per_unit': 3.86,      # FBA 配送费
        'storage_per_unit': 0.15,      # 月仓储费/件
        'ad_spend_monthly': 1200,      # 月广告花费
        'return_rate': 0.08,           # 退货率 8%
        'return_cost_per_unit': 5.00,  # 退货处理费/件
        'tool_cost_monthly': 99,       # 工具订阅月费
    },
    {
        'sku': 'SKU-002',
        'name': 'Phone Stand Adjustable',
        'selling_price': 19.99,
        'monthly_units': 800,
        'cogs_per_unit': 3.20,
        'shipping_per_unit': 0.80,
        'tariff_rate': 0.03,
        'referral_fee_rate': 0.15,
        'fba_fee_per_unit': 3.22,
        'storage_per_unit': 0.10,
        'ad_spend_monthly': 600,
        'return_rate': 0.05,
        'return_cost_per_unit': 3.50,
        'tool_cost_monthly': 0,
    },
    {
        'sku': 'SKU-003',
        'name': 'LED Desk Lamp',
        'selling_price': 29.99,
        'monthly_units': 300,
        'cogs_per_unit': 7.00,
        'shipping_per_unit': 2.50,
        'tariff_rate': 0.04,
        'referral_fee_rate': 0.15,
        'fba_fee_per_unit': 5.10,
        'storage_per_unit': 0.25,
        'ad_spend_monthly': 800,
        'return_rate': 0.06,
        'return_cost_per_unit': 6.00,
        'tool_cost_monthly': 0,
    }
]

print(f'Loaded {len(products)} SKUs')

## 3. 计算真实利润

In [ ]:
results = []
for p in products:
    units = p['monthly_units']
    revenue = p['selling_price'] * units
    
    # 成本明细
    cogs = p['cogs_per_unit'] * units
    shipping = p['shipping_per_unit'] * units
    tariff = p['cogs_per_unit'] * p['tariff_rate'] * units
    referral = revenue * p['referral_fee_rate']
    fba = p['fba_fee_per_unit'] * units
    storage = p['storage_per_unit'] * units
    ads = p['ad_spend_monthly']
    returns = units * p['return_rate'] * p['return_cost_per_unit']
    lost_revenue = units * p['return_rate'] * p['selling_price']  # 退货损失的收入
    tools = p['tool_cost_monthly']
    
    total_cost = cogs + shipping + tariff + referral + fba + storage + ads + returns + tools
    net_revenue = revenue - lost_revenue
    net_profit = net_revenue - total_cost
    margin = net_profit / revenue * 100
    profit_per_unit = net_profit / units
    
    results.append({
        'SKU': p['sku'],
        'Product': p['name'],
        'Price': p['selling_price'],
        'Units/Mo': units,
        'Revenue': revenue,
        'COGS': cogs,
        'Shipping': shipping,
        'Tariff': tariff,
        'Referral Fee': referral,
        'FBA Fee': fba,
        'Storage': storage,
        'Ad Spend': ads,
        'Return Cost': returns,
        'Tools': tools,
        'Total Cost': total_cost,
        'Net Profit': net_profit,
        'Margin %': margin,
        'Profit/Unit': profit_per_unit
    })

df = pd.DataFrame(results)
print('=== SKU 真实利润分析 ===')
for _, row in df.iterrows():
    emoji = '✅' if row['Margin %'] > 15 else ('⚠️' if row['Margin %'] > 5 else '❌')
    print(f"\n{emoji} {row['Product']} ({row['SKU']})")
    print(f"   Revenue: ${row['Revenue']:,.0f} | Profit: ${row['Net Profit']:,.0f} | Margin: {row['Margin %']:.1f}%")
    print(f"   Profit/Unit: ${row['Profit/Unit']:.2f}")

print(f"\n{'='*50}")
print(f"Total Monthly Revenue: ${df['Revenue'].sum():,.0f}")
print(f"Total Monthly Profit: ${df['Net Profit'].sum():,.0f}")
print(f"Overall Margin: {df['Net Profit'].sum() / df['Revenue'].sum() * 100:.1f}%")

## 4. 成本分解可视化

In [ ]:
# 利润瀑布图（第一个 SKU）
p = results[0]
categories = ['Revenue', 'COGS', 'Shipping', 'Tariff', 'Referral', 'FBA', 'Storage', 'Ads', 'Returns', 'Tools', 'Profit']
values = [p['Revenue'], -p['COGS'], -p['Shipping'], -p['Tariff'], -p['Referral Fee'], -p['FBA Fee'], -p['Storage'], -p['Ad Spend'], -p['Return Cost'], -p['Tools'], p['Net Profit']]
measures = ['absolute'] + ['relative'] * 9 + ['total']

fig = go.Figure(go.Waterfall(
    x=categories, y=values, measure=measures,
    connector={'line': {'color': 'rgb(63, 63, 63)'}},
    increasing={'marker': {'color': 'green'}},
    decreasing={'marker': {'color': 'red'}},
    totals={'marker': {'color': 'blue'}}
))
fig.update_layout(title=f'Profit Waterfall: {p["Product"]}', yaxis_title='USD')
fig.show()

# 成本结构饼图
cost_items = ['COGS', 'Shipping', 'Tariff', 'Referral Fee', 'FBA Fee', 'Storage', 'Ad Spend', 'Return Cost', 'Tools']
cost_values = [p[item] for item in cost_items]
fig = px.pie(values=cost_values, names=cost_items, title=f'Cost Breakdown: {p["Product"]}', hole=0.3)
fig.show()

# SKU 利润对比
fig = px.bar(df, x='Product', y=['Net Profit', 'Total Cost'], barmode='group',
             title='SKU Profit vs Cost Comparison')
fig.show()

## 5. 价格敏感度分析

In [ ]:
# 价格变化对利润的影响
p = products[0]
print(f'=== Price Sensitivity: {p["name"]} ===')
print(f'Current price: ${p["selling_price"]}')
print()

for change in [-20, -15, -10, -5, 0, 5, 10, 15, 20]:
    new_price = p['selling_price'] * (1 + change/100)
    new_revenue = new_price * p['monthly_units']
    # Simplified: assume same costs except referral fee
    new_referral = new_revenue * p['referral_fee_rate']
    base_cost = (p['cogs_per_unit'] + p['shipping_per_unit'] + p['fba_fee_per_unit'] + p['storage_per_unit']) * p['monthly_units']
    new_profit = new_revenue * (1 - p['return_rate']) - base_cost - new_referral - p['ad_spend_monthly'] - p['tool_cost_monthly']
    new_margin = new_profit / new_revenue * 100
    emoji = '📈' if change > 0 else ('📉' if change < 0 else '➡️')
    print(f'{emoji} {change:+d}% → ${new_price:.2f} | Profit: ${new_profit:,.0f} | Margin: {new_margin:.1f}%')

## 6. 导出

In [ ]:
df.to_csv('sku_profit_analysis.csv', index=False)
print('Exported to sku_profit_analysis.csv')
print('\nNext: Use this data to optimize pricing (A8) and ad spend (A3)')